# Thử nghiệm học máy cho mục 3.6

Notebook cập nhật Feature Mart từ các cột đã có trong application_train rồi so sánh bộ đặc trưng hiện tại với bộ mở rộng trên cùng cách chia dữ liệu. Notebook chạy lại hai mô hình dbt cần thiết; thao tác này cập nhật bảng Gold Feature Mart, không sửa dữ liệu nguồn. Kết quả được ghi riêng trong `exports/ml_experiment/`.

Phạm vi: mô hình cơ sở dự đoán lớp phổ biến nhất và hai Logistic Regression (đặc trưng hiện tại, đặc trưng mở rộng); chia tập có phân tầng giống nhau; xử lý thiếu và mã hóa chỉ học trên tập huấn luyện; chọn ngưỡng riêng trên tập kiểm định; đánh giá tập giữ lại một lần. Mã hồ sơ, TARGET và code_gender không được dùng làm đặc trưng.

In [8]:
%pip -q install duckdb==1.5.6 pandas scikit-learn joblib dbt-core==1.9.4 dbt-duckdb==1.9.4

import sys, subprocess
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform
import duckdb
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, brier_score_loss, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_CANDIDATES = [Path('/content/drive/MyDrive/Data_Lakehouse_Project')]
else:
    HERE = Path.cwd()
    PROJECT_CANDIDATES = [HERE, HERE / 'Data_Lakehouse_Project', HERE.parent / 'Data_Lakehouse_Project']
PROJECT = next((p for p in PROJECT_CANDIDATES if (p / 'dbt').is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError('Không tìm thấy thư mục dự án có thư mục dbt. Hãy đặt dự án tại MyDrive/Data_Lakehouse_Project trên Colab.')
DB_CANDIDATES = [PROJECT / 'database/credit_lakehouse.duckdb', PROJECT / 'data/lakehouse_db/credit_lakehouse.duckdb']
DB_PATH = next((p for p in DB_CANDIDATES if p.is_file()), None)
if DB_PATH is None:
    raise FileNotFoundError('Không tìm thấy credit_lakehouse.duckdb ở database/ hoặc data/lakehouse_db/.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = PROJECT / 'exports' / 'ml_experiment' / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
SEED = 42

# Dựng lại hai mô hình đã cập nhật vào đúng cơ sở dữ liệu của dự án.
DBT_DIR = PROJECT / 'dbt'
PROFILE_DIR = OUT / 'dbt_profiles'
TARGET_DIR = OUT / 'dbt_target'
PROFILE_DIR.mkdir(parents=True, exist_ok=True)
TARGET_DIR.mkdir(parents=True, exist_ok=True)
profiles_yaml = (
    'credit_lakehouse:\n'
    '  target: local\n'
    '  outputs:\n'
    '    local:\n'
    '      type: duckdb\n'
    f'      path: {json.dumps(str(DB_PATH))}\n'
    '      threads: 4\n'
)
(PROFILE_DIR / 'profiles.yml').write_text(profiles_yaml, encoding='utf-8')
subprocess.run([
    'dbt', 'run', '--select', 'int_application_train_prep', 'feature_mart_application_train',
    '--project-dir', str(DBT_DIR), '--profiles-dir', str(PROFILE_DIR), '--target-path', str(TARGET_DIR)
], check=True)
print('Dự án:', PROJECT)
print('Cơ sở dữ liệu:', DB_PATH)
print('Thư mục kết quả lần chạy:', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dự án: /content/drive/MyDrive/Data_Lakehouse_Project
Cơ sở dữ liệu: /content/drive/MyDrive/Data_Lakehouse_Project/database/credit_lakehouse.duckdb
Thư mục kết quả lần chạy: /content/drive/MyDrive/Data_Lakehouse_Project/exports/ml_experiment/20261106T203859Z


## 1. Dựng và kiểm tra Feature Mart

Notebook chạy lại hai mô hình dbt để đưa thêm các trường sẵn có trong application_train vào Feature Mart, sau đó chỉ đọc bảng Gold vừa cập nhật. Kiểm tra dừng sớm nếu sai bảng/cột, khóa thiếu hoặc trùng, nhãn ngoài {0,1}, hoặc không đủ hai lớp. Mã hồ sơ, TARGET và giới tính không tham gia tập đặc trưng.

In [9]:
con = duckdb.connect(str(DB_PATH), read_only=True)
tables = con.execute("select table_schema, table_name from information_schema.tables where table_schema='gold'").fetchdf()
if not ((tables.table_name == 'feature_mart_application_train').any()):
    raise ValueError('Không thấy gold.feature_mart_application_train. Hãy chạy lại dbt để tạo Feature Mart trước.')
df = con.execute('select * from gold.feature_mart_application_train').fetchdf()
con.close()
required = {'sk_id_curr', 'target'}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f'Thiếu cột bắt buộc trong Feature Mart: {sorted(missing_required)}')
if df.empty:
    raise ValueError('Feature Mart không có dữ liệu.')
if df['sk_id_curr'].isna().any() or df['sk_id_curr'].duplicated().any():
    raise ValueError('Khóa sk_id_curr có giá trị thiếu hoặc trùng.')
if df['target'].isna().any() or not set(pd.to_numeric(df['target'], errors='coerce').dropna().unique()).issubset({0, 1}):
    raise ValueError('target phải không thiếu và chỉ gồm 0, 1.')
df['target'] = df['target'].astype('int8')
class_counts = df['target'].value_counts().sort_index()
if len(class_counts) != 2 or class_counts.min() < 10:
    raise ValueError('Cần đủ hai lớp và tối thiểu 10 hồ sơ mỗi lớp để chia tập phân tầng ổn định.')
base_candidate_features = [
    'amt_credit', 'amt_income_total', 'amt_annuity', 'age_years',
    'cleaned_days_employed', 'flag_days_employed_anomaly',
    'name_contract_type', 'credit_to_income', 'annuity_to_income'
]
expanded_candidate_features = base_candidate_features + [
    'amt_goods_price', 'cnt_children', 'cnt_fam_members', 'days_registration',
    'days_id_publish', 'days_last_phone_change', 'region_population_relative',
    'region_rating_client', 'region_rating_client_w_city', 'ext_source_1', 'ext_source_2', 'ext_source_3',
    'obs_30_cnt_social_circle', 'def_30_cnt_social_circle', 'obs_60_cnt_social_circle', 'def_60_cnt_social_circle',
    'amt_req_credit_bureau_hour', 'amt_req_credit_bureau_day', 'amt_req_credit_bureau_week',
    'amt_req_credit_bureau_mon', 'amt_req_credit_bureau_qrt', 'amt_req_credit_bureau_year',
    'name_income_type', 'name_education_type', 'name_family_status', 'name_housing_type',
    'occupation_type', 'name_type_suite', 'flag_own_car', 'flag_own_realty',
    'credit_to_goods_price', 'income_per_family_member', 'employment_years'
]
features = [c for c in expanded_candidate_features if c in df.columns]
base_features = [c for c in base_candidate_features if c in df.columns]
missing_candidates = [c for c in expanded_candidate_features if c not in df.columns]
if not features or not base_features:
    raise ValueError('Thiếu cột đặc trưng dự kiến. Kiểm tra kết quả dbt run và schema Feature Mart.')
excluded_present = [c for c in ['target', 'sk_id_curr', 'source_sk_id_curr', 'code_gender'] if c in df.columns]
if set(features) & set(excluded_present):
    raise AssertionError('Phát hiện cột nhãn/nhận dạng/giới tính trong danh sách đặc trưng.')
X = df[features].copy()
y = df['target'].copy()
numeric_features = X.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_features = [c for c in features if c not in numeric_features]
for c in numeric_features:
    X[c] = pd.to_numeric(X[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
all_missing_features = X.columns[X.isna().all()].tolist()
if all_missing_features:
    X = X.drop(columns=all_missing_features)
    features = X.columns.tolist()
    base_features = [c for c in base_features if c in features]
    numeric_features = [c for c in numeric_features if c in features]
    categorical_features = [c for c in categorical_features if c in features]
if not features or not base_features:
    raise ValueError('Không còn đủ đặc trưng có dữ liệu để so sánh.')
X_base = X[base_features].copy()
quality = pd.DataFrame({'dtype': X.dtypes.astype(str), 'missing_count': X.isna().sum(), 'missing_rate': X.isna().mean()})
quality.to_csv(OUT / 'feature_quality.csv', encoding='utf-8-sig')
print('Số hồ sơ:', len(df))
print('Phân bố nhãn:', class_counts.to_dict())
print('Số đặc trưng hiện tại:', len(base_features), base_features)
print('Số đặc trưng mở rộng:', len(features), features)
print('Cột dự kiến không có trong mart:', missing_candidates)
print('Cột toàn thiếu đã loại:', all_missing_features)
print('Cột loại khỏi mô hình:', excluded_present)
display(quality)

Số hồ sơ: 307511
Phân bố nhãn: {0: 282686, 1: 24825}
Số đặc trưng hiện tại: 9 ['amt_credit', 'amt_income_total', 'amt_annuity', 'age_years', 'cleaned_days_employed', 'flag_days_employed_anomaly', 'name_contract_type', 'credit_to_income', 'annuity_to_income']
Số đặc trưng mở rộng: 42 ['amt_credit', 'amt_income_total', 'amt_annuity', 'age_years', 'cleaned_days_employed', 'flag_days_employed_anomaly', 'name_contract_type', 'credit_to_income', 'annuity_to_income', 'amt_goods_price', 'cnt_children', 'cnt_fam_members', 'days_registration', 'days_id_publish', 'days_last_phone_change', 'region_population_relative', 'region_rating_client', 'region_rating_client_w_city', 'ext_source_1', 'ext_source_2', 'ext_source_3', 'obs_30_cnt_social_circle', 'def_30_cnt_social_circle', 'obs_60_cnt_social_circle', 'def_60_cnt_social_circle', 'amt_req_credit_bureau_hour', 'amt_req_credit_bureau_day', 'amt_req_credit_bureau_week', 'amt_req_credit_bureau_mon', 'amt_req_credit_bureau_qrt', 'amt_req_credit_bureau_

,dtype,missing_count,missing_rate
amt_credit,float64,0,0.000000
amt_income_total,float64,0,0.000000
amt_annuity,float64,12,0.000039
age_years,int32,0,0.000000
cleaned_days_employed,Int32,55374,0.180072
flag_days_employed_anomaly,int32,0,0.000000
name_contract_type,object,0,0.000000
credit_to_income,float64,0,0.000000
annuity_to_income,float64,12,0.000039
amt_goods_price,float64,278,0.000904


## 2. Chia tập và kiểm soát rò rỉ

Hai bộ đặc trưng dùng đúng cùng một lần chia: khoảng 70% huấn luyện, 15% kiểm định và 15% giữ lại; phân tầng theo TARGET. Mã hồ sơ chỉ dùng để xác nhận các tập rời nhau. Bộ xử lý thiếu, chuẩn hóa và mã hóa nằm trong quy trình mô hình nên chỉ được khớp trên tập huấn luyện.

In [10]:
ids_trainval, ids_holdout, y_trainval, y_holdout = train_test_split(
    df['sk_id_curr'], y, test_size=0.15, random_state=SEED, stratify=y)
X_trainval = X.loc[ids_trainval.index]
X_holdout = X.loc[ids_holdout.index]
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval, y_trainval, test_size=(0.15 / 0.85), random_state=SEED, stratify=y_trainval)
ids_train, ids_valid = ids_trainval.loc[X_train.index], ids_trainval.loc[X_valid.index]
if set(ids_train) & set(ids_valid) or set(ids_train) & set(ids_holdout) or set(ids_valid) & set(ids_holdout):
    raise AssertionError('Có hồ sơ xuất hiện ở nhiều tập dữ liệu.')
split_summary = pd.DataFrame([
    {'split':'train','rows':len(y_train),'target_0':int((y_train==0).sum()),'target_1':int((y_train==1).sum())},
    {'split':'validation','rows':len(y_valid),'target_0':int((y_valid==0).sum()),'target_1':int((y_valid==1).sum())},
    {'split':'holdout','rows':len(y_holdout),'target_0':int((y_holdout==0).sum()),'target_1':int((y_holdout==1).sum())},
])
split_summary.to_csv(OUT / 'split_summary.csv', index=False, encoding='utf-8-sig')
display(split_summary)

,split,rows,target_0,target_1
0,train,215257,197880,17377
1,validation,46127,42403,3724
2,holdout,46127,42403,3724


## 3. Huấn luyện và chọn ngưỡng

Huấn luyện hai Logistic Regression: một dùng bộ đặc trưng hiện tại, một dùng bộ mở rộng. Cả hai dùng cùng cách chia; ngưỡng được chọn riêng để tối đa F1 trên tập kiểm định. Không xem tập giữ lại để chọn mô hình hoặc ngưỡng.

In [11]:
numeric_pipe = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
categorical_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))])
def build_pipeline(columns):
    nums = [c for c in columns if c in numeric_features]
    cats = [c for c in columns if c in categorical_features]
    transformers = []
    if nums: transformers.append(('numeric', numeric_pipe, nums))
    if cats: transformers.append(('categorical', categorical_pipe, cats))
    preprocess = ColumnTransformer(transformers=transformers, remainder='drop')
    return Pipeline([('preprocess', preprocess), ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED))])
base_model = build_pipeline(base_features)
expanded_model = build_pipeline(features)
base_model.fit(X_train[base_features], y_train)
expanded_model.fit(X_train[features], y_train)
base_valid_prob = base_model.predict_proba(X_valid[base_features])[:, 1]
expanded_valid_prob = expanded_model.predict_proba(X_valid[features])[:, 1]
threshold_grid = np.arange(0.05, 0.951, 0.01)
def choose_f1_threshold(y_true, probabilities):
    scores = [f1_score(y_true, probabilities >= t, zero_division=0) for t in threshold_grid]
    index = int(np.argmax(scores))
    return float(threshold_grid[index]), float(scores[index])
base_threshold, base_valid_f1 = choose_f1_threshold(y_valid, base_valid_prob)
expanded_threshold, expanded_valid_f1 = choose_f1_threshold(y_valid, expanded_valid_prob)
print(f'Bộ hiện tại: ngưỡng={base_threshold:.2f}, F1 kiểm định={base_valid_f1:.4f}')
print(f'Bộ mở rộng: ngưỡng={expanded_threshold:.2f}, F1 kiểm định={expanded_valid_f1:.4f}')

Bộ hiện tại: ngưỡng=0.54, F1 kiểm định=0.1934
Bộ mở rộng: ngưỡng=0.66, F1 kiểm định=0.2939


## 4. Đánh giá cuối trên tập giữ lại

So sánh mô hình cơ sở, Logistic Regression với bộ đặc trưng hiện tại và Logistic Regression với bộ mở rộng. Các chỉ số phân loại dùng ngưỡng chọn riêng trên tập kiểm định. Đây là kết quả thử nghiệm trên dữ liệu hiện có, không phải quyết định phê duyệt tín dụng.

In [12]:
def evaluate(name, y_true, prob, threshold):
    pred = (prob >= threshold).astype('int8')
    return {'model':name, 'threshold':float(threshold), 'rows':int(len(y_true)),
        'roc_auc':float(roc_auc_score(y_true, prob)), 'pr_auc':float(average_precision_score(y_true, prob)),
        'precision':float(precision_score(y_true, pred, zero_division=0)),
        'recall':float(recall_score(y_true, pred, zero_division=0)), 'f1':float(f1_score(y_true, pred, zero_division=0)),
        'brier':float(brier_score_loss(y_true, prob)), 'tn_fp_fn_tp':confusion_matrix(y_true, pred, labels=[0,1]).ravel().tolist()}
dummy = DummyClassifier(strategy='most_frequent').fit(np.zeros((len(y_train),1)), y_train)
dummy_prob = dummy.predict_proba(np.zeros((len(y_holdout),1)))[:, 1]
base_holdout_prob = base_model.predict_proba(X_holdout[base_features])[:, 1]
expanded_holdout_prob = expanded_model.predict_proba(X_holdout[features])[:, 1]
results = [
    evaluate('baseline_majority', y_holdout, dummy_prob, 0.5),
    evaluate('logistic_regression_current_features', y_holdout, base_holdout_prob, base_threshold),
    evaluate('logistic_regression_expanded_features', y_holdout, expanded_holdout_prob, expanded_threshold)
]
metrics = pd.DataFrame(results)
metrics.to_csv(OUT / 'holdout_metrics.csv', index=False, encoding='utf-8-sig')
display(metrics)

,model,threshold,rows,roc_auc,pr_auc,precision,recall,f1,brier,tn_fp_fn_tp
0,baseline_majority,0.50,46127,0.500000,0.080734,0.000000,0.000000,0.000000,0.080734,"[42403, 0, 3724, 0]"
1,logistic_regression_current_features,0.54,46127,0.615949,0.116771,0.116811,0.481203,0.187988,0.239698,"[28854, 13549, 1932, 1792]"
2,logistic_regression_expanded_features,0.66,46127,0.746312,0.223846,0.228762,0.404941,0.292361,0.204889,"[37319, 5084, 2216, 1508]"


## 5. Kiểm tra tham khảo theo nhóm

Nếu Feature Mart có `code_gender`, notebook chỉ dùng cột này sau khi dự báo để tính chỉ số tổng hợp theo nhóm; cột này không tham gia huấn luyện. Nhóm quá nhỏ được ghi là không đủ cỡ mẫu. So sánh này chỉ là kiểm tra ban đầu, không chứng minh mô hình công bằng.

In [13]:
if 'code_gender' in df.columns:
    audit = pd.DataFrame({'group':df.loc[X_holdout.index, 'code_gender'].astype('string').fillna('MISSING'),
        'y':y_holdout.to_numpy(), 'prob':expanded_holdout_prob})
    audit_rows = []
    for group, part in audit.groupby('group', dropna=False):
        pred = (part['prob'] >= expanded_threshold).astype('int8')
        enough = len(part) >= 100 and part['y'].nunique() == 2
        audit_rows.append({'group':str(group), 'rows':int(len(part)), 'positive_rate':float(part['y'].mean()),
            'roc_auc':float(roc_auc_score(part['y'], part['prob'])) if enough else np.nan,
            'recall':float(recall_score(part['y'], pred, zero_division=0)) if enough else np.nan,
            'false_positive_rate':float(((pred==1)&(part['y']==0)).sum() / max((part['y']==0).sum(),1)) if enough else np.nan,
            'status':'Đủ điều kiện mô tả' if enough else 'Cỡ mẫu/lớp chưa đủ'})
    subgroup = pd.DataFrame(audit_rows)
    subgroup.to_csv(OUT / 'subgroup_audit_gender.csv', index=False, encoding='utf-8-sig')
    display(subgroup)
else:
    print('Không có code_gender trong Feature Mart; bỏ qua kiểm tra nhóm này.')

,group,rows,positive_rate,roc_auc,recall,false_positive_rate,status
0,F,30390,0.069727,0.737541,0.363379,0.101305,Đủ điều kiện mô tả
1,M,15735,0.102002,0.748023,0.459813,0.157042,Đủ điều kiện mô tả
2,XNA,2,0.000000,NaN,NaN,NaN,Cỡ mẫu/lớp chưa đủ


## 6. Lưu lần chạy có thể tái lập

Tệp kê khai ghi nhận dấu vân tay của dữ liệu dùng trong thử nghiệm, bảng nguồn, danh sách đặc trưng, cách chia tập, hạt giống ngẫu nhiên, cấu hình và phiên bản thư viện. Mô hình đã huấn luyện được lưu riêng; không lưu dự báo theo từng hồ sơ. Đây là ghi nhận cho một thử nghiệm, chưa phải hệ thống quản lý mô hình hay MLOps đầy đủ.

In [14]:
# Dấu vân tay nội dung bảng: băm ID, nhãn và các đặc trưng theo thứ tự khóa.
fingerprint_frame = df[['sk_id_curr', 'target'] + features].sort_values('sk_id_curr').reset_index(drop=True)
row_hashes = pd.util.hash_pandas_object(fingerprint_frame, index=False).to_numpy(dtype='uint64')
data_fingerprint = hashlib.sha256(row_hashes.tobytes()).hexdigest()
joblib.dump(expanded_model, OUT / 'logistic_regression_expanded_pipeline.joblib')
manifest = {
    'run_id': RUN_ID, 'created_utc': datetime.now(timezone.utc).isoformat(),
    'database_file': DB_PATH.name, 'database_size_bytes': DB_PATH.stat().st_size,
    'database_modified_utc': datetime.fromtimestamp(DB_PATH.stat().st_mtime, timezone.utc).isoformat(),
    'source_relation': 'gold.feature_mart_application_train', 'row_count': int(len(df)),
    'data_fingerprint_sha256': data_fingerprint, 'target': 'target',
    'base_features': base_features, 'expanded_features': features,
    'excluded_from_features': excluded_present, 'missing_candidate_features': missing_candidates,
    'all_missing_features_dropped': all_missing_features, 'seed': SEED,
    'split': {'train':0.70, 'validation':0.15, 'holdout':0.15, 'method':'stratified random', 'key':'sk_id_curr'},
    'threshold_selection': {'method':'maximum F1 on validation',
        'current_features':base_threshold, 'expanded_features':expanded_threshold},
    'models': ['DummyClassifier(most_frequent)',
        'LogisticRegression current features (class_weight=balanced, max_iter=2000)',
        'LogisticRegression expanded features (class_weight=balanced, max_iter=2000)'],
    'versions': {'python':platform.python_version(), 'duckdb':duckdb.__version__, 'pandas':pd.__version__, 'scikit_learn':sklearn.__version__},
    'limitations': ['TARGET là nhãn của bộ dữ liệu, không khẳng định tương đương nợ xấu theo quy định.',
        'Đánh giá ngẫu nhiên trên cùng nguồn dữ liệu; chưa phải kiểm định theo thời gian hoặc triển khai thực tế.',
        'Không dùng code_gender để huấn luyện; kiểm tra nhóm nếu có chỉ mang tính tham khảo.',
        'Kết quả không tự động quyết định phê duyệt tín dụng.']
}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu kết quả thực nghiệm tại:', OUT)
print('Các tệp:', sorted(p.name for p in OUT.iterdir()))

Đã lưu kết quả thực nghiệm tại: /content/drive/MyDrive/Data_Lakehouse_Project/exports/ml_experiment/20261106T203859Z
Các tệp: ['dbt_profiles', 'dbt_target', 'feature_quality.csv', 'holdout_metrics.csv', 'logistic_regression_expanded_pipeline.joblib', 'run_manifest.json', 'split_summary.csv', 'subgroup_audit_gender.csv']
